# Custom Models for `did_multiplegt_stat`

This notebook demonstrates using custom ML models via `model_deltay` and `model_stayer`,
inspired by DoubleML's `ml_g` / `ml_m` pattern.

- `model_deltay`: model for E[DeltaY | D1, S=0] (outcome regression, replaces OLS)
- `model_stayer`: model for P(stayer | D1) (propensity score, replaces logit)

Any sklearn-compatible estimator with `.fit(X, y)` and `.predict(X)` / `.predict_proba(X)` works.

In [1]:
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
from did_multiplegt_stat import did_multiplegt_stat

df = pd.read_stata('gazoline_did_multiplegt_stat.dta')
print(f'Gazoline: {df.shape}')

Gazoline: (2064, 354)


In [2]:
def extract_all(result):
    """Extract main estimates from result dict."""
    out = {}
    tbl = result.get('results', {}).get('table', None)
    if isinstance(tbl, pd.DataFrame):
        for idx in tbl.index:
            out[idx] = tbl.iloc[tbl.index.get_loc(idx), 0]
    return out

## Example 1: RandomForest (both outcome and propensity)

In [3]:
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier

print('Running with RandomForest models...')
r_rf = did_multiplegt_stat(df, 'lngca', 'id', 'year', 'tau',
                           order=1, controls=['lngpinc'],
                           model_deltay=RandomForestRegressor(n_estimators=100, random_state=42),
                           model_stayer=RandomForestClassifier(n_estimators=100, random_state=42))
print('Done.')

Running with RandomForest models...


Done.


## Example 2: LassoCV (outcome only, default logit for propensity)

In [4]:
from sklearn.linear_model import LassoCV

print('Running with LassoCV for outcome...')
r_lasso = did_multiplegt_stat(df, 'lngca', 'id', 'year', 'tau',
                              order=1, controls=['lngpinc'],
                              model_deltay=LassoCV())
print('Done.')

Running with LassoCV for outcome...


Done.


## Example 3: Default sklearn (asinstata=False)

In [5]:
print('Running with default sklearn models...')
r_default = did_multiplegt_stat(df, 'lngca', 'id', 'year', 'tau',
                                order=1, controls=['lngpinc'])
print('Done.')

Running with default sklearn models...


Done.


## Example 4: Stata-faithful (asinstata=True)

In [6]:
print('Running with asinstata=True...')
r_stata = did_multiplegt_stat(df, 'lngca', 'id', 'year', 'tau',
                              order=1, controls=['lngpinc'],
                              asinstata=True)
print('Done.')

Running with asinstata=True...


Done.


## Comparison Table

In [7]:
est_rf = extract_all(r_rf)
est_lasso = extract_all(r_lasso)
est_default = extract_all(r_default)
est_stata = extract_all(r_stata)

all_keys = sorted(set(list(est_rf.keys()) + list(est_lasso.keys()) +
                      list(est_default.keys()) + list(est_stata.keys())))

rows = []
for k in all_keys:
    rows.append({
        'Estimator': k,
        'RandomForest': est_rf.get(k, np.nan),
        'LassoCV': est_lasso.get(k, np.nan),
        'Default (sklearn)': est_default.get(k, np.nan),
        'asinstata=True': est_stata.get(k, np.nan),
    })

comp = pd.DataFrame(rows)
comp.index = comp['Estimator']
comp = comp.drop(columns='Estimator')

print('='*80)
print('  Comparison: 4 model configurations')
print('='*80)
display(comp)

  Comparison: 4 model configurations


,RandomForest,LassoCV,Default (sklearn),asinstata=True
Estimator,,,,
AS,-0.000159,-0.001266,-0.001383,-0.001384
IV-WAS,NaN,NaN,NaN,NaN
WAS,-0.003147,-0.003661,-0.003754,-0.003756
as_10,-0.002782,-0.002404,-0.002407,-0.002407
as_11,0.004005,0.003401,0.003483,0.003483
...,...,...,...,...
was_5,0.005941,-0.003903,-0.003830,-0.003830
was_6,-0.003854,0.008211,0.008187,0.008187
was_7,-0.008137,-0.007037,-0.006903,-0.006903


In [8]:
# Difference from asinstata=True baseline
baseline = comp['asinstata=True']
diff_pct = comp.apply(lambda col: ((col - baseline) / baseline.abs() * 100)
                      .where(baseline.abs() > 0), axis=0)

print('\nDifference from asinstata=True (%):')
display(diff_pct.round(4))


Difference from asinstata=True (%):


,RandomForest,LassoCV,Default (sklearn),asinstata=True
Estimator,,,,
AS,88.5167,8.5160,0.0229,0.0
IV-WAS,NaN,NaN,NaN,NaN
WAS,16.2164,2.5316,0.0553,0.0
as_10,-15.5967,0.1300,0.0028,0.0
as_11,14.9790,-2.3616,0.0013,0.0
...,...,...,...,...
was_5,255.1074,-1.9073,0.0020,0.0
was_6,-147.0689,0.2816,-0.0001,0.0
was_7,-17.8821,-1.9481,0.0001,0.0
